# 7th Merge Method: Magnitude-Calibrated Merge (CT-Merging-inspired)

**Motivation.** Every prior method (linear/svd/ties/dare/slerp/bwsum) assumes equal per-adapter weight (1/3 each) means equal per-adapter *influence*. Measured directly on these three adapters, that's false — codealpaca's ΔW has ~5x dolly's Frobenius norm and ~2x metamath's, and linear merge's cosine retention to each adapter (0.20 / 0.46 / 0.88) tracks that norm ratio almost exactly. Codealpaca silently dominates every additive merge method tried so far.

**Fix.** Rescale each adapter's ΔW to a common target magnitude *before* combining, so equal coefficients actually mean equal influence. This was validated on the raw deltas earlier (cosine retention went from 0.20/0.46/0.88 to a balanced 0.56/0.61/0.60) — this notebook merges, uploads, and evaluates it end-to-end for the first time.

Note: earlier this was described as a 'concatenation' method — that framing was incorrect. Concatenating LoRA A/B factors and then combining collapses mathematically to the same weighted sum as linear merge for a linear layer; it changes nothing on its own. The real fix is the magnitude calibration, not concatenation.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Load Adapters

In [ ]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import json
import torch

def get_lora_deltas(repo_name: str, hf_username: str = "Srishtik", lora_alpha: int = None, r: int = None) -> dict:
    """Compute ΔW = lora_B @ lora_A * (alpha/r). r/alpha read from the adapter's
    own adapter_config.json unless overridden — never assume a shared config."""
    repo_id = f"{hf_username}/{repo_name}"
    config_path = hf_hub_download(repo_id=repo_id, filename="adapter_config.json")
    cfg = json.load(open(config_path))

    actual_r     = r if r is not None else cfg.get("r")
    actual_alpha = lora_alpha if lora_alpha is not None else cfg.get("lora_alpha")
    if actual_r is None or actual_alpha is None:
        raise ValueError(f"Could not determine r/lora_alpha for {repo_name}: {cfg}")

    path = hf_hub_download(repo_id=repo_id, filename="adapter_model.safetensors")
    adapter_weights = load_file(path)
    scale = actual_alpha / actual_r

    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()

    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])

    print(f"  {repo_name:<35} r={actual_r}  alpha={actual_alpha}  scale={scale:.4f}  layers={len(deltas)}")
    return deltas


print("Loading adapter deltas:")
dolly_deltas      = get_lora_deltas("qwen3-trained-on-dolly-15k")
metamath_deltas   = get_lora_deltas("qwen3-trained-on-metamath-15k")
codealpaca_deltas = get_lora_deltas("qwen3-trained-on-code-alpaca-18k")

adapter_names = ["dolly", "metamath", "codealpaca"]
deltas = [dolly_deltas, metamath_deltas, codealpaca_deltas]


## Load Base Model, Normalize Keys

In [ ]:
from unsloth import FastLanguageModel
import torch

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-0.6B",
    max_seq_length = 2048,
    load_in_4bit   = False,
    dtype          = torch.float16,
)
base_sd = base_model.state_dict()
print(f"Base keys: {len(base_sd)}")
print(f"Sample base keys: {list(base_sd.keys())[:3]}")

del base_model
torch.cuda.empty_cache()


def normalize_delta_keys(deltas: dict) -> dict:
    """Convert 'base_model.model.model.layers.0.self_attn.q_proj.' to
    'model.layers.0.self_attn.q_proj.weight'"""
    normalized = {}
    for k, v in deltas.items():
        new_key = k.replace("base_model.model.", "")
        new_key = new_key.rstrip(".")
        new_key = new_key + ".weight"
        normalized[new_key] = v
    return normalized

dolly_deltas      = normalize_delta_keys(dolly_deltas)
metamath_deltas   = normalize_delta_keys(metamath_deltas)
codealpaca_deltas = normalize_delta_keys(codealpaca_deltas)
deltas = [dolly_deltas, metamath_deltas, codealpaca_deltas]

print(f"Sample normalized key: {list(dolly_deltas.keys())[:3]}")
print(f"Overlap with base_sd: {len(set(dolly_deltas.keys()) & set(base_sd.keys()))}")  # expect 196


def apply_delta_to_base(base_state_dict: dict, delta_state_dict: dict) -> dict:
    merged = {}
    for key, base_val in base_state_dict.items():
        if key in delta_state_dict:
            delta_val = delta_state_dict[key].to(base_val.device)  # match device before adding
            merged[key] = (base_val.float() + delta_val).to(base_val.dtype)
        else:
            merged[key] = base_val
    return merged


## The 7th Method: Magnitude-Calibrated Merge

In [ ]:
import torch

def total_norm(deltas: dict) -> float:
    """Aggregate Frobenius norm across all layers of one adapter's delta."""
    return sum(torch.norm(v.float()).item() ** 2 for v in deltas.values()) ** 0.5


def ct_calibrated_merge(deltas: list, weights: list = None, target: str = "mean") -> dict:
    """
    7th merge method: magnitude-calibrated linear merge (CT-Merging-inspired).

    MOTIVATION: linear/svd/ties/dare/bwsum all assume equal per-adapter weight
    (1/3 each) is equivalent to equal per-adapter INFLUENCE. Measured directly on
    this experiment's adapters, that assumption is false: codealpaca's ΔW has
    ~5x dolly's Frobenius norm and ~2x metamath's. Plain linear merge's cosine
    retention (dolly=0.20, metamath=0.46, codealpaca=0.88) tracks that norm
    ratio almost exactly — codealpaca silently dominates every additive method.

    FIX: rescale each adapter's ΔW to a common target magnitude BEFORE combining,
    so equal coefficients actually mean equal influence. Validated empirically:
    this takes cosine retention from (0.20, 0.46, 0.88) to a balanced
    (0.56, 0.61, 0.60) across all three adapters.

    Unlike "concatenation" framings of this idea, this does NOT change the
    merged rank (still rank-16, same footprint as linear/ties/dare) — the fix
    is purely in the per-adapter scale factor applied before summation.

    Args:
        deltas  : list of per-adapter ΔW state dicts
        weights : post-calibration combination weights (default: equal, 1/n each)
        target  : "mean" (arithmetic mean of the adapters' norms) or
                  "geometric" (geometric mean — more conservative, always <= mean)
    """
    n = len(deltas)
    weights = weights or [1.0 / n] * n

    norms = [total_norm(d) for d in deltas]

    if target == "mean":
        target_norm = sum(norms) / n
    elif target == "geometric":
        target_norm = 1.0
        for norm in norms:
            target_norm *= norm
        target_norm = target_norm ** (1.0 / n)
    else:
        raise ValueError(f"Unknown target: {target}")

    print(f"  Original norms : {[round(x, 4) for x in norms]}")
    print(f"  Target norm ({target}): {target_norm:.4f}")
    print(f"  Rescale factors: {[round(target_norm / x, 4) for x in norms]}")

    rescaled = []
    for d, orig_norm in zip(deltas, norms):
        scale = target_norm / orig_norm
        rescaled.append({k: v * scale for k, v in d.items()})

    keys = set.intersection(*[set(d.keys()) for d in rescaled])
    merged = {k: sum(w * d[k] for w, d in zip(weights, rescaled)) for k in keys}
    return merged


## Apply Merge

In [ ]:
merged_delta_ct = ct_calibrated_merge(deltas, weights=[1/3, 1/3, 1/3], target="mean")
merged_sd_ct = apply_delta_to_base(base_sd, merged_delta_ct)
print(f"\nMerged state dict ready: {len(merged_sd_ct)} keys")


## Upload to HuggingFace

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

def upload_merged_model(
    merged_sd: dict,
    repo_name: str,
    tokenizer,
    hf_token: str,
    base_repo: str = "unsloth/Qwen3-0.6B",
    max_seq_length: int = 2048,
    dtype = torch.float16,
    push_to_hub: bool = True,
):
    print(f"[upload] Preparing model → {repo_name}")

    model, _ = FastLanguageModel.from_pretrained(
        model_name     = base_repo,
        max_seq_length = max_seq_length,
        load_in_4bit   = False,
        dtype          = dtype,
    )

    target_dtype = next(model.parameters()).dtype
    cast_sd = {k: v.to(target_dtype) if v.is_floating_point() else v for k, v in merged_sd.items()}

    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys   : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")

    model.eval()

    if push_to_hub:
        model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")

    del model, cast_sd
    torch.cuda.empty_cache()
    return repo_name


HF_TOKEN = "your_hf_token_here"  # Insert your own token

CT_REPO_NAME = "Srishtik/Qwen3-0.6B-ct-calibrated-3-adapters-merged-2"
uploaded_ct_repo = upload_merged_model(
    merged_sd  = merged_sd_ct,
    repo_name  = CT_REPO_NAME,
    tokenizer  = tokenizer,
    hf_token   = HF_TOKEN,
)


## Evaluation

Same pipeline as the main eval notebook: GSM8K exact match (n=200), HumanEval pass@1 (n=164, `enable_thinking=False` fix applied), Dolly-15k perplexity (n=200, response-tokens-only NLL).

In [ ]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def prep_tokenizer_for_generation(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# ── GSM8K ──
def extract_gsm8k_answer(text: str) -> str:
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()

def format_gsm8k_prompt(question: str) -> str:
    return (f"Solve the following math problem. Show your reasoning and put "
            f"your final numeric answer after '#### '.\n\nQuestion: {question}")

def evaluate_gsm8k(model, tokenizer, model_name="model", num_samples=200, batch_size=4,
                    max_new_tokens=320, device="cuda"):
    print(f"\n{'─'*60}\n[GSM8K] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    preds, labels = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]
        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.3)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            preds.append(extract_gsm8k_answer(generated))
            labels.append(true_answers[j])
    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match = round(sum(per_sample_exact) / len(per_sample_exact), 4)
    print(f"  Exact Match: {exact_match:.4f}")
    return {"repo_id": model_name, "exact_match": exact_match, "num_samples": len(per_sample_exact),
            "per_sample_exact": per_sample_exact}

# ── HumanEval ──
def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    return problem_prompt + "\n" + text

def _unsafe_execute(program: str, result_list, timeout: int):
    import signal
    def handler(signum, frame):
        raise TimeoutError("execution timed out")
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")

def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    program = completion_code + "\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill(); p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"

def format_humaneval_prompt(problem_prompt: str) -> str:
    return ("Complete the following Python function. Return ONLY the complete "
            "function code (including the signature), with no explanations and "
            f"no markdown formatting.\n\n{problem_prompt}")

def evaluate_humaneval(model, tokenizer, model_name="model", num_samples=164, batch_size=4,
                        max_new_tokens=384, device="cuda"):
    print(f"\n{'─'*60}\n[HumanEval] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    per_sample_pass = []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]
        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=768, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.1)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code = extract_code(generated, problem_prompts[j], entry_points[j])
            problem = {"prompt": problem_prompts[j], "test": batch["test"][j], "entry_point": entry_points[j]}
            per_sample_pass.append(int(check_correctness(problem, code, timeout=5)))
    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)
    print(f"  pass@1: {pass_at_1:.4f}")
    return {"repo_id": model_name, "pass_at_1": pass_at_1, "num_samples": len(per_sample_pass),
            "per_sample_pass": per_sample_pass}

# ── Dolly-15k perplexity ──
def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"

def evaluate_dolly_perplexity(model, tokenizer, model_name="model", num_samples=200,
                               max_length=512, device="cuda"):
    print(f"\n{'─'*60}\n[Dolly-PPL] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    per_sample_nll, per_sample_ppl = [], []
    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100
        with torch.no_grad():
            out = model(full_ids, labels=labels)
        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))
    result = {"repo_id": model_name, "perplexity": round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
              "mean_nll": round(sum(per_sample_nll) / len(per_sample_nll), 4),
              "num_samples": len(per_sample_ppl), "per_sample_nll": per_sample_nll}
    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")
    return result


In [ ]:
eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = uploaded_ct_repo,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(eval_model)

ct_gsm8k     = evaluate_gsm8k(eval_model, eval_tokenizer, model_name=uploaded_ct_repo,
                               num_samples=200, batch_size=4)
ct_humaneval = evaluate_humaneval(eval_model, eval_tokenizer, model_name=uploaded_ct_repo,
                                   num_samples=164, batch_size=4)
ct_dolly     = evaluate_dolly_perplexity(eval_model, eval_tokenizer, model_name=uploaded_ct_repo,
                                          num_samples=200)

del eval_model, eval_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"\n{'═'*60}")
print(f"ct-calibrated-merge summary:")
print(f"  GSM8K Exact Match : {ct_gsm8k['exact_match']:.4f}")
print(f"  HumanEval pass@1  : {ct_humaneval['pass_at_1']:.4f}")
print(f"  Dolly Perplexity  : {ct_dolly['perplexity']:.4f}")
print(f"{'═'*60}")


## Comparison Against Existing 6 Methods + Specialists

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Comparison against previously measured results (from prior evaluation runs —
# hardcoded here rather than re-running 10+ models, since those numbers are
# already established; only the new method needed a fresh run).
# ─────────────────────────────────────────────────────────────────────────────

known_results = {
    "linear"             : {"gsm8k": 0.1100, "humaneval": 0.2012, "dolly_ppl": 17.8096},
    "svd"                : {"gsm8k": 0.1550, "humaneval": 0.2073, "dolly_ppl": 20.5070},
    "ties"                : {"gsm8k": 0.0550, "humaneval": 0.2073, "dolly_ppl": 16.7428},
    "dare"                : {"gsm8k": 0.0400, "humaneval": 0.2378, "dolly_ppl": 17.2781},
    "bwsum"               : {"gsm8k": 0.1200, "humaneval": 0.1829, "dolly_ppl": 18.8800},
    "slerp (buggy, best order)": {"gsm8k": 0.0100, "humaneval": 0.2073, "dolly_ppl": 13.1499},
    "codealpaca_adapter (specialist)": {"gsm8k": 0.0850, "humaneval": 0.1707, "dolly_ppl": 38.3633},
    "metamath_adapter (specialist)"  : {"gsm8k": 0.2200, "humaneval": 0.1220, "dolly_ppl": 28.6676},
    "dolly_adapter (specialist)"     : {"gsm8k": 0.0250, "humaneval": 0.1646, "dolly_ppl": 12.6278},
}

print(f"{'Model':<38}{'GSM8K':>10}{'HumanEval':>12}{'Dolly PPL':>12}")
print("-" * 72)
for name, r in known_results.items():
    print(f"{name:<38}{r['gsm8k']:>10.4f}{r['humaneval']:>12.4f}{r['dolly_ppl']:>12.4f}")

print("-" * 72)
print(f"{'ct-calibrated (NEW, 7th method)':<38}{ct_gsm8k['exact_match']:>10.4f}"
      f"{ct_humaneval['pass_at_1']:>12.4f}{ct_dolly['perplexity']:>12.4f}")
print("=" * 72)

print("\nRanking check — how does the new method compare to the best of the")
print("existing 6 merge methods on each task?")
best_gsm8k     = max(known_results.items(), key=lambda x: x[1]["gsm8k"] if "specialist" not in x[0] else -1)
best_humaneval = max(known_results.items(), key=lambda x: x[1]["humaneval"] if "specialist" not in x[0] else -1)
best_dolly     = min(known_results.items(), key=lambda x: x[1]["dolly_ppl"] if "specialist" not in x[0] else float("inf"))

print(f"  GSM8K     — best prior merge method: {best_gsm8k[0]} ({best_gsm8k[1]['gsm8k']:.4f})  "
      f"vs ct-calibrated: {ct_gsm8k['exact_match']:.4f}")
print(f"  HumanEval — best prior merge method: {best_humaneval[0]} ({best_humaneval[1]['humaneval']:.4f})  "
      f"vs ct-calibrated: {ct_humaneval['pass_at_1']:.4f}")
print(f"  Dolly PPL — best prior merge method: {best_dolly[0]} ({best_dolly[1]['dolly_ppl']:.4f})  "
      f"vs ct-calibrated: {ct_dolly['perplexity']:.4f}")
